# SoccerMaster — example notebook

This walks through the refactored `soccermaster` package: install it, fetch the
pretrained weights, load the model, run inference, and optionally run the full
game-state pipeline.

It is written to run on **Google Colab** (a GPU runtime is recommended) and should
also work on Kaggle and on a local machine.

## What is verified

Everything up to and including inference has been run end to end on a Colab T4
(Python 3.13.15, torch 2.11.0+cu130, transformers 4.51.3):

- `pip install` from GitHub, including the `pipeline` extra
- `import soccermaster` and its paths
- `load_config()` and the weight manifest
- downloading the SigLIP2 backbone + SoccerMaster checkpoints
- `load_model()` — 967.8 M parameters, all five heads, onto `cuda:0`
- `predict(..., task="detection")` — returns `pred_logits`, `pred_boxes`,
  `pred_roles`, `pred_jn_holistic`, `pred_digit_head`, `pred_digit_tail`, `outputs`

**Not verified:** the 3-step game-state pipeline (section 8) has not been run. It
needs the `yolo`/`qwen7b` weights plus the SoccerNet-hosted `reid` and
`calibration` sets, none of which are in the manifest as automatic downloads.
Expect to fix small things on its first real run.

Cells are **guarded** so nothing multi-gigabyte downloads or runs by accident.

## Prerequisite

You need access to the weights. `soccermaster prepare` handles the Hugging Face and
Google Drive ones; the `reid` and `calibration` sets are SoccerNet-hosted and must be
placed by hand (see the last section).

## If something breaks

This fork is new and the code base is large. Most first-run failures so far have been
dependency versions rather than model logic — see the troubleshooting section at the
end, which lists every one hit while getting this notebook to pass.


In [35]:
# Install the package.
#
# `pipeline` is the extra that carries everything the game-state steps import at
# module level (torchreid, prtreid, SoccerNet, poseval, pycocotools, ...). Without it
# you can still import the package and load the model, but the pipeline will raise
# ModuleNotFoundError. Use "" for the lean install.
#
# This cell installs from the LOCAL checkout when there is one, so unpushed fixes are
# the thing being tested. Otherwise it installs from GitHub, which is what you want on
# Colab/Kaggle.
#
# CAUTION on hosted runtimes: they ship a CUDA-matched torch, and a dependency with an
# upper torch bound would downgrade it and break CUDA. That is why this project pins
# floors only (torch>=2.4). Nothing here force-reinstalls without --no-deps, so the
# runtime's own torch is never touched.

import importlib.util
import subprocess
import sys
from pathlib import Path

GIT_URL = "https://github.com/SurrealisticRabbit/SoccerMaster.git"
GIT_REF = "main"
EXTRAS = "pipeline"  # "" for the lean install, "pipeline,sam2" for step 2 too

# Is this notebook running inside a checkout of the project?
ROOT = None
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate = parent / "pyproject.toml"
    if candidate.is_file() and "soccermaster" in candidate.read_text(errors="ignore"):
        ROOT = parent
        break

if ROOT is not None:
    SPEC = f"{ROOT}[{EXTRAS}]" if EXTRAS else str(ROOT)
else:
    SPEC = f"soccermaster[{EXTRAS}] @ git+{GIT_URL}@{GIT_REF}"

ALREADY = importlib.util.find_spec("soccermaster") is not None

print(f"python        : {sys.executable} ({sys.version.split()[0]})")
print(f"local checkout: {ROOT}")
print(f"already there : {ALREADY}")
print(f"installing    : {SPEC}\n")


def pip(*args):
    """Run pip and show what it did; raise with the real error on failure."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-1500:])
    if result.returncode:
        print(result.stderr[-3000:])
        raise RuntimeError(f"pip failed (exit {result.returncode}): {' '.join(cmd)}")


if ROOT is not None:
    # editable install of the checkout; dependencies resolve normally
    pip("-e", SPEC)
elif ALREADY:
    # pip will not replace an already-installed direct URL whose version string is
    # unchanged, so a plain re-run would silently keep the old code. Force the code in
    # first, without touching dependencies (and so without touching torch)...
    pip("--force-reinstall", "--no-deps", "--upgrade", SPEC)
    # ...then install normally so any newly-declared dependency is picked up.
    pip("--upgrade", SPEC)
else:
    pip("--upgrade", SPEC)

print("done")


python        : /usr/bin/python3 (3.13.15)
local checkout: None
already there : True
installing    : soccermaster[pipeline] @ git+https://github.com/SurrealisticRabbit/SoccerMaster.git@main

SoccerMaster.git@main (from soccermaster[pipeline]@ git+https://github.com/SurrealisticRabbit/SoccerMaster.git@main)
  Cloning https://github.com/SurrealisticRabbit/SoccerMaster.git (to revision main) to /tmp/pip-install-xr16txd4/soccermaster_691b0f116bf7426aaf890280b755eeee
  Resolved https://github.com/SurrealisticRabbit/SoccerMaster.git to commit de918938f0273b61e56839bdff8b35e02bc24465
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for soccermaster: filename=soccermaster-0.1.0-py3-none-any.whl size=10

## 1. Environment and where things get stored

Everything the package reads and writes lives under a single root —
`SOCCERMASTER_HOME`. You must set it **before** the paths are first used so that
weights and outputs land somewhere you can find (and, on Colab, somewhere you know
is ephemeral unless you mount Drive).

Run this cell, then **restart the runtime** if you set a new value, so every
subsequent import sees it.


In [7]:
import os
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Where should weights/, data/ and outputs/ live?
# ---------------------------------------------------------------------------
USE_DRIVE = False  # set True on Colab to persist weights across sessions

try:
    import google.colab  # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if USE_DRIVE and ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    SM_HOME = Path("/content/drive/MyDrive/soccermaster")
elif ON_COLAB:
    # /content is wiped when the runtime recycles -- fine for a one-off run.
    SM_HOME = Path("/content/soccermaster")
else:
    SM_HOME = Path.cwd() / "soccermaster_home"

SM_HOME.mkdir(parents=True, exist_ok=True)
os.environ["SOCCERMASTER_HOME"] = str(SM_HOME)

print(f"python      : {sys.version.split()[0]}")
print(f"on colab    : {ON_COLAB}")
print(f"SM_HOME     : {SM_HOME}")

try:
    import torch

    print(f"torch       : {torch.__version__}")
    print(f"cuda avail  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"gpu         : {torch.cuda.get_device_name(0)}")
except ImportError:
    print("torch       : NOT INSTALLED -- install a CUDA build before continuing")


python      : 3.13.15
on colab    : True
SM_HOME     : /content/soccermaster
torch       : 2.11.0+cu130
cuda avail  : True
gpu         : Tesla T4


## 2. Verify the install

`import soccermaster` is deliberately dependency-free: the public API is re-exported
lazily, so this import succeeds even if nothing else is installed. That makes it a
quick check that the package itself landed.


In [8]:
import soccermaster as sm

print("soccermaster :", sm.__version__)
print("home         :", sm.paths.home())
print("weights      :", sm.paths.weights_dir())
print("data         :", sm.paths.data_dir())
print("outputs      :", sm.paths.outputs_dir())

# The public surface: everything a notebook should need.
print("api          :", ", ".join(sm.__all__))


soccermaster : 0.1.0
home         : /content/soccermaster
weights      : /content/soccermaster/weights
data         : /content/soccermaster/data
outputs      : /content/soccermaster/outputs
api          : load_config, load_model, predict, prepare, prepare_all, run_pipeline, paths


## 3. What weights are needed

`soccermaster prepare` reads a manifest and puts each set where the configs expect
it. Two entries cannot be fetched automatically and say so.


In [9]:
from soccermaster.downloads import load_manifest

manifest = load_manifest()
print(f"{'name':<14}{'kind':<8}{'source':<9}destination")
print("-" * 78)
for name, entry in manifest.items():
    kind = "auto" if entry.automatic else "MANUAL"
    print(f"{name:<14}{kind:<8}{entry.source:<9}{entry.dest}")

# Which of them are already on disk?
print("\nAlready present:")
for name, entry in manifest.items():
    mark = "yes" if entry.dest.exists() and any(entry.dest.iterdir()) else "no "
    print(f"  {name:<14}{mark}")


name          kind    source   destination
------------------------------------------------------------------------------
siglip        auto    hf       /content/soccermaster/weights/google/siglip2-large-patch16-512
soccermaster  auto    hf       /content/soccermaster/weights/SoccerMaster
yolo          auto    hf       /content/soccermaster/weights/yolo
qwen7b        auto    hf       /content/soccermaster/weights/jn/Qwen2.5-VL-7B-Instruct
qwen72b       auto    hf       /content/soccermaster/weights/jn/Qwen2.5-VL-72B-Instruct
sam2          auto    hf       /content/soccermaster/weights/sam2
legibility    auto    gdrive   /content/soccermaster/weights/legibility
reid          MANUAL  manual   /content/soccermaster/weights/reid
calibration   MANUAL  manual   /content/soccermaster/weights/calibration

Already present:
  siglip        no 
  soccermaster  no 
  yolo          no 
  qwen7b        no 
  qwen72b       no 
  sam2          no 
  legibility    no 
  reid          no 
  calibration 

## 4. Download the weights

Multi-gigabyte. The cell is guarded by `DOWNLOAD`; the `dry_run` pass below prints
exactly what would be fetched without touching the network.

For **inference only** you need `siglip` (backbone) + `soccermaster` (checkpoints).
Add `yolo` for step 1 of the pipeline, `qwen7b` for step 3, `sam2` for step 2.


In [10]:
DOWNLOAD = False  # flip to True to actually fetch the weights

# What would happen, without downloading anything:
for name in ("siglip", "soccermaster", "yolo"):
    sm.prepare(name, dry_run=True)

if DOWNLOAD:
    sm.prepare("siglip")        # SigLIP2 backbone (Google) -- needed to build the model
    sm.prepare("soccermaster")  # SoccerMaster checkpoints  -- needed for inference
    sm.prepare("yolo")          # person detector          -- needed for pipeline step 1
    sm.prepare("qwen7b")        # jersey/role (Alibaba)    -- needed for pipeline step 3
    # sm.prepare("sam2")        # SAM 2 checkpoint         -- needed for pipeline step 2
else:
    print("DOWNLOAD is False - nothing fetched.")


[siglip] source=hf dest=/content/soccermaster/weights/google/siglip2-large-patch16-512
  SigLIP2 backbone used by the SoccerMaster encoder.
  would download hf://google/siglip2-large-patch16-512 (allow=*)
[soccermaster] source=hf dest=/content/soccermaster/weights/SoccerMaster
  SoccerMaster pretrained checkpoints (backbone + task heads).
  would download hf://xleprime/SoccerMaster (allow=*)
[yolo] source=hf dest=/content/soccermaster/weights/yolo
  Person detector used by the GSR step-1 pipeline.
  would download hf://xleprime/SoccerMaster (allow=['yolo_v8x6_finetuned.pt', 'yolo/*'])
DOWNLOAD is False - nothing fetched.


## 5. Inspect the config (no weights needed)

Cheap sanity check. This also shows how paths are resolved: `CKPT_PATH` is rewritten
to point under `weights_dir()`, so the configs no longer depend on the working
directory.


In [11]:
cfg = sm.load_config("pretrain")

print(f"config keys : {len(cfg)}")
for key in (
    "SIGLIP_BACKBONE_TYPE",
    "NUM_FRAMES",
    "AUG_MAX_SIZE",
    "TEMPORAL_START_LAYER",
    "CKPT_PATH",
    "TEXT_ENCODER_CKPT_PATH",
):
    print(f"{key:<22}: {cfg.get(key)}")

print("\nDATASETS_TO_HEADS (task groups -> heads):")
for dataset, heads in cfg["DATASETS_TO_HEADS"].items():
    print(f"  {dataset:<24} {heads}")


config keys : 182
SIGLIP_BACKBONE_TYPE  : soccer_master
NUM_FRAMES            : 30
AUG_MAX_SIZE          : 512
TEMPORAL_START_LAYER  : 16
CKPT_PATH             : /content/soccermaster/weights/google/siglip2-large-patch16-512
TEXT_ENCODER_CKPT_PATH: /content/soccermaster/weights/google/siglip2-large-patch16-512

DATASETS_TO_HEADS (task groups -> heads):
  SoccerNetGSR_Detection   ['SoccerNetGSR_Detection', 'LinesDetection', 'KeypointsDetection']
  VideoCaption             ['VideoCaption', 'CaptionClassification']


## 6. Load the model

`load_model()` builds `MultiTaskingModel` and loads the backbone plus the task heads
from `weights/SoccerMaster`. Guarded so the cell explains itself instead of raising a
confusing error when the weights are absent.


In [37]:
model = None

checkpoint_dir = sm.paths.weights_dir() / "SoccerMaster"
backbone_dir = cfg["CKPT_PATH"]

missing = [p for p in (checkpoint_dir, backbone_dir) if not Path(p).exists()]

if missing:
    print("Weights not found yet:")
    for p in missing:
        print(f"  {p}")
    print("\nSet DOWNLOAD = True in the download cell and re-run it, or place the")
    print("files manually. The rest of the notebook needs these.")
else:
    model = sm.load_model()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"model      : {type(model).__name__}")
    print(f"parameters : {n_params/1e6:.1f} M")
    print(f"device     : {next(model.parameters()).device}")
    print(f"heads      : {list(model.multi_task_head.keys())}")


Using Backbone type: soccer_master
Loaded backbone weights from: /content/soccermaster/weights/SoccerMaster/backbone.pt
Loading CaptionClassification head from: /content/soccermaster/weights/SoccerMaster/CaptionClassification.pt
Loading KeypointsDetection head from: /content/soccermaster/weights/SoccerMaster/KeypointsDetection.pt
Loading LinesDetection head from: /content/soccermaster/weights/SoccerMaster/LinesDetection.pt
Loading SoccerNetGSR_Detection head from: /content/soccermaster/weights/SoccerMaster/SoccerNetGSR_Detection.pt
Loading VideoCaption head from: /content/soccermaster/weights/SoccerMaster/VideoCaption.pt
model      : MultiTaskingModel
parameters : 967.8 M
device     : cuda:0
heads      : ['CaptionClassification', 'KeypointsDetection', 'LinesDetection', 'SoccerNetGSR_Detection', 'VideoCaption']


## 7. Run inference

`predict(model, frames, task=...)` takes either:

- a directory — `.../SNGS-10001` or `.../SNGS-10001/img1` (a flat folder of images
  also works), or
- an array of `uint8` HWC frames, or a list of such arrays.

It returns the **raw head outputs** (no NMS, no bbox rescaling), matching the
upstream `inference_demo.py`. `task` is one of `detection`, `lines`, `keypoints`,
`caption`, `classification` or `all`.

The cell below uses random frames so you can exercise the forward pass without any
dataset. Point `FRAMES` at a real sequence to get meaningful output.


In [41]:
import numpy as np

# Point this at a real sequence when you have one, e.g.
#   sm.paths.data_dir() / "SoccerNetGS" / "sn500" / "SNGS-10001"
FRAMES = None

if model is None:
    print("Load the model first (previous cell).")
else:
    if FRAMES is None:
        # The temporal encoder expects cfg["NUM_FRAMES"] frames.
        n = cfg["NUM_FRAMES"]
        FRAMES = np.random.randint(0, 256, size=(n, 540, 960, 3), dtype=np.uint8)
        print(f"using {n} random frames at {FRAMES.shape[1]}x{FRAMES.shape[2]}")

    try:
        outputs = sm.predict(model, FRAMES, task="detection")
    except Exception as exc:  # surface the real error, this path is unverified
        print(f"predict failed: {type(exc).__name__}: {exc}")
    else:
        def describe(value, indent="    "):
            import torch

            if isinstance(value, torch.Tensor):
                print(f"{indent}shape={tuple(value.shape)} dtype={value.dtype}")
            elif isinstance(value, dict):
                for k, v in value.items():
                    print(f"{indent}{k}:")
                    describe(v, indent + "  ")
            elif isinstance(value, (list, tuple)):
                print(f"{indent}{type(value).__name__} of length {len(value)}")
                for i, item in enumerate(value[:2]):
                    print(f"{indent}  [{i}]")
                    describe(item, indent + "    ")
            else:
                print(f"{indent}{value}")

        for head, head_output in outputs.items():
            print(f"[{head}]")
            describe(head_output)


using 30 random frames at 540x960
[SoccerNetGSR_Detection]
    pred_logits:
      shape=(1, 30, 300, 1) dtype=torch.float32
    pred_boxes:
      shape=(1, 30, 300, 4) dtype=torch.float32
    pred_roles:
      shape=(1, 30, 300, 6) dtype=torch.float32
    pred_jn_holistic:
      shape=(1, 30, 300, 101) dtype=torch.float32
    pred_digit_head:
      shape=(1, 30, 300, 10) dtype=torch.float32
    pred_digit_tail:
      shape=(1, 30, 300, 11) dtype=torch.float32
    aux_outputs:
      list of length 0
    outputs:
      shape=(1, 30, 300, 256) dtype=torch.float32


## 8. The full game-state pipeline (optional)

Three steps:

| step | what it does | needs |
|---|---|---|
| 1 | person detection + tracking | `yolo` weights |
| 2 | SAM 2 mask refinement | `sam2` extra + `sam2` weights + CUDA |
| 3 | calibration, jersey number, role, team | `qwen7b` weights, `reid` + `calibration` weights |

This is the same Hydra configuration the upstream scripts used, driven for you: the
dataset path, the weight directory and the state files between steps are injected, so
it no longer depends on the working directory or on hardcoded sequence ids.

Steps 1 and 3 run on CPU or GPU. **Step 2 requires the `sam2` extra and a CUDA
build.** The dataset must be in SoccerNet layout:
`<dataset>/<split>/<sequence>/img1/*.jpg` plus `sequences_info.json`.


In [42]:
RUN_PIPELINE = False  # flip to True once the dataset and weights are in place

DATASET = sm.paths.data_dir() / "SoccerNetGS"   # <split>/<sequence>/img1 layout
OUT_DIR = sm.paths.outputs_dir() / "pipeline"
VIDEO_IDS = [10001, 10002]                      # sequences to process
STEPS = (1, 3)                                  # skip 2 unless the sam2 extra is installed

if RUN_PIPELINE:
    if not DATASET.is_dir():
        print(f"dataset not found: {DATASET}")
        print("Run: python -m soccermaster.model.data.extract_frames --help")
    else:
        # device is a CUDA_VISIBLE_DEVICES value, e.g. "0" or "0,1"
        out = sm.run_pipeline(
            DATASET,
            out_dir=OUT_DIR,
            steps=STEPS,
            video_ids=VIDEO_IDS,
            device="0",
        )
        print("outputs written to:", out)
        for path in sorted(out.rglob("*.pklz")):
            print(" ", path.relative_to(out), f"({path.stat().st_size/1e6:.1f} MB)")
else:
    print("RUN_PIPELINE is False - nothing run.")
    print(f"would read  : {DATASET}")
    print(f"would write : {OUT_DIR}")


RUN_PIPELINE is False - nothing run.
would read  : /content/soccermaster/data/SoccerNetGS
would write : /content/soccermaster/outputs/pipeline


## Troubleshooting

Every failure below was hit for real while getting this notebook to run on Colab.
Most were wrong dependency versions, not model logic.

**`ERROR: Could not find a version that satisfies the requirement torchvision<0.21,>=0.19`**
An upper bound on torchvision makes the package uninstallable: hosted runtimes only
have torchvision >= 0.21. Fixed — the project pins floors only now. If you are on an
older commit, install from a newer `main`.

**CUDA stopped working after the install**
Reinstall with `--no-deps` and provide the dependencies yourself, or pin the torch you
want first. The install cell in this notebook deliberately avoids touching torch.

**`AttributeError: 'SiglipVisionModel' object has no attribute 'vision_model'`**
transformers 5.x (and later 4.x) flattened `SiglipVisionModel`; it exposes
`.embeddings` / `.encoder.layers` / `.post_layernorm` / `.head` directly. Fixed in the
model code, and transformers is now capped below 4.52.

**`TypeError: ... smart_apply() missing 1 required positional argument: 'is_custom_code'`**
transformers 4.57.6 itself is broken for `SiglipVisionModel.from_pretrained`. 4.51.3
is the verified-good version and is what the pin resolves to.

**`ModuleNotFoundError: No module named 'transformers.models.diffusion_gemma'`**
Downgrading transformers in place on Colab leaves files behind from the newer
release. Uninstall before installing:
`pip uninstall -y transformers && pip install "transformers==4.51.3"`.

**`ImportError: cannot import name 'LsqEllipse' from 'ellipse'`**
The distribution providing `from ellipse import LsqEllipse` is `lsq-ellipse`, not
`ellipse`. Fixed in the dependencies.

**`ModuleNotFoundError: No module named 'MultiScaleDeformableAttention'`**
That is the CUDA-compiled Deformable DETR op. It is now optional: the code falls back
to a pure-PyTorch implementation, so the model loads without a CUDA toolchain (slower
per forward pass). Build the extension for speed if you need it.

**`RuntimeError: The size of tensor a (576) must match the size of tensor b (1024)`**
The encoder's positional embedding is a fixed 32x32 patch grid, so inputs must be
square at `AUG_MAX_SIZE` (512). `predict()` now resizes to a square for you — which
distorts aspect ratio, so predicted boxes are in the resized frame.

**Stale code after reinstalling**
The kernel keeps already-imported modules in memory. If behaviour does not match the
code on disk, purge them:
```python
import sys
for m in [m for m in list(sys.modules) if m.split(".")[0] == "soccermaster"]:
    del sys.modules[m]
```

**`ModuleNotFoundError: No module named 'torchreid'` (or `prtreid`, `SoccerNet`, `poseval`, `pycocotools`)**
You installed without the `pipeline` extra. The base install only covers import +
model; the pipeline deps live in `.[pipeline]`.

**`FileNotFoundError` for `siglip2-large-patch16-512` or `SoccerMaster`**
The weights are not there. Set `DOWNLOAD = True` in the download cell and re-run it,
or place the files under `sm.paths.weights_dir()`.

**Step 1 complains about the YOLO checkpoint**
The configs look for `weights/yolo/yolo_v8x6_person_lr_default_best.pt`. The manifest
fetches `yolo_v8x6_finetuned.pt` from Hugging Face and renames it to that name; check
the file actually landed.

**`reid` / `calibration` are missing**
These are `manual` in the manifest and print upstream instructions rather than
downloading. They come from the
[SoccerNet GSR dev kit](https://github.com/soccernet/sn-gamestate). Put them under
`weights/reid/` and `weights/calibration/`.

**Step 2 will not start**
Step 2 needs the `sam2` extra *and* the vendored SAM 2 library:
```bash
pip install -e ".[sam2]"
pip install -e third_party/sam2     # only in a source checkout
```
and a CUDA build. On a CPU-only runtime, run `steps=(1, 3)` instead.

**Where do I look when something breaks?**
`PROVENANCE.md` maps every folder to its upstream project. If a failure is inside
`tracklab/`, `soccernet_gamestate/` or `sam2_refine/`, it is vendored third-party code
and the upstream project or its own docs in `docs/` is the better reference.

## Verified versions

```
python 3.13.15  (Colab)
torch 2.11.0+cu130          (the runtime's own; the project pins >=2.4 only)
transformers 4.51.3         (pinned >=4.49,<4.52)
```

## Useful bits

```bash
soccermaster --help          # list subcommands
soccermaster prepare --list  # the weight manifest
soccermaster pipeline --dataset ... --steps 1,3 --videos 10001
soccermaster predict --frames path/to/img1 --task detection
```

`run_pipeline` forwards extra Hydra overrides, so anything in the upstream configs can
still be tweaked, e.g. `extra_overrides=["dataset.nframes=20"]`.
